# DQN Simple - Tu Primer Deep Q-Network

¡Bienvenido! Este notebook es el **puente** entre Q-Learning tabular y Deep RL avanzado.

## ¿Por qué necesitamos DQN?

**Problema con Q-Tables**:
- FrozenLake: 16 estados → Q-table de 16×4 ✅ Funciona
- CartPole: Estados continuos → Q-table infinita ❌ Imposible

**Solución: Usar una Red Neuronal**
- En lugar de tabla: $Q(s,a)$ → Red neuronal: $Q(s,a; \theta)$
- La red **aproxima** los Q-values
- Puede generalizar a estados nunca vistos

## Al final de este notebook, podrás:

✅ Crear una red Q simple con PyTorch

✅ Hacer predicciones de Q-values

✅ Calcular loss y entrenar la red

✅ Entrenar un agente DQN básico en CartPole

**Duración estimada**: 1 hora ⏱️

**Prerequisito**: Haber completado `pytorch_for_rl.ipynb`

## Tabla de Contenidos
- [1 - Preparación](#1)
- [2 - Red Q Simple](#2)
    - [Ejercicio 1 - crear_red_q](#ex-1)
- [3 - Predicción de Q-values](#3)
    - [Ejercicio 2 - predecir_q_values](#ex-2)
- [4 - Calcular Loss](#4)
    - [Ejercicio 3 - calcular_dqn_loss](#ex-3)
- [5 - Entrenar un Paso](#5)
    - [Ejercicio 4 - entrenar_paso_dqn](#ex-4)
- [6 - Loop de Entrenamiento](#6)
    - [Ejercicio 5 - entrenar_dqn_simple](#ex-5)
- [7 - ¡Felicitaciones!](#7)

## 📖 Teoría: De Q-Tables a Redes Neuronales

### El Problema de Escala

**Q-Learning Tabular**:
$$Q: S \times A \rightarrow \mathbb{R}$$

Guardamos Q-values en una tabla:

| Estado | Acción 0 | Acción 1 |
|--------|----------|----------|
| s₀ | 0.5 | 0.3 |
| s₁ | 0.8 | 0.2 |
| ... | ... | ... |

**Problema**: 
- FrozenLake: 16 estados → Tabla de 16×4 = 64 valores ✅
- CartPole: Estados continuos (posición, velocidad, ángulo, vel. angular) → ∞ estados ❌
- Atari: 210×160×3 píxeles → ~10⁶⁷⁰⁰⁰ estados posibles ❌❌❌

### La Solución: Aproximación de Funciones

En lugar de tabla, usamos una **función aproximadora**:
$$Q(s,a) \approx Q(s,a; \theta)$$

Donde:
- $\theta$ = parámetros de la red neuronal (pesos y biases)
- La red **generaliza** a estados nunca vistos

### Ecuación de Bellman con Redes Neuronales

**Q-Learning Tabular**:
$$Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$$

**DQN (Deep Q-Network)**:
$$\theta \leftarrow \theta - \alpha \nabla_\theta L(\theta)$$

Donde el loss es:
$$L(\theta) = \mathbb{E}_{(s,a,r,s')}\left[(r + \gamma \max_{a'} Q(s',a';\theta) - Q(s,a;\theta))^2\right]$$

**Diferencias clave**:
- Tabular: Actualiza **un valor** en la tabla
- DQN: Actualiza **todos los pesos** de la red usando gradientes

### ¿Por qué funciona?

1. **Generalización**: Si la red aprende que `Q(s₁, a₁) = 0.8`, puede inferir que estados similares a `s₁` también tienen Q-values altos
2. **Eficiencia**: No necesitamos visitar todos los estados
3. **Escalabilidad**: Funciona con espacios de estados enormes

### Arquitectura Simple

```
Estado (vector) → [Red Neuronal] → Q-values (vector)
     [x₁, x₂, x₃, x₄]  →  [fc1 → ReLU → fc2]  →  [Q(s,a₀), Q(s,a₁)]
```

**Ejemplo CartPole**:
- Input: `[posición, velocidad, ángulo, vel_angular]` (4 valores)
- Output: `[Q(s, izquierda), Q(s, derecha)]` (2 valores)

---

**¡Ahora vamos a implementarlo paso a paso!**


### 📚 Notación Matemática

Para que puedas seguir las fórmulas, aquí está la notación que usamos:

| Símbolo | Significado | Ejemplo |
|---------|-------------|----------|
| $s$ | Estado (state) | `[posición, velocidad, ángulo, vel_angular]` |
| $a$ | Acción (action) | `0` (izquierda) o `1` (derecha) |
| $r$ | Recompensa (reward) | `+1` por cada timestep |
| $s'$ | Siguiente estado (next state) | Estado después de tomar acción $a$ |
| $a'$ | Siguiente acción | Acción en el siguiente estado |
| $Q(s,a)$ | Q-value | Valor esperado de tomar acción $a$ en estado $s$ |
| $\theta$ | Parámetros de la red | Pesos y biases de la red neuronal |
| $\alpha$ | Learning rate | Qué tan rápido aprende (ej: 0.01) |
| $\gamma$ | Factor de descuento (gamma) | Importancia del futuro (ej: 0.99) |
| $\max_{a'}$ | Máximo sobre acciones | El Q-value más alto entre todas las acciones |
| $\nabla_\theta$ | Gradiente respecto a $\theta$ | Derivadas para actualizar pesos |
| $\mathbb{E}$ | Valor esperado (expectation) | Promedio sobre muchas experiencias |
| $L(\theta)$ | Función de loss | Qué tan mal están las predicciones |

### Ecuaciones Clave Explicadas

**1. Q-Learning Tabular**:
$$Q(s,a) \leftarrow Q(s,a) + \alpha [\underbrace{r + \gamma \max_{a'} Q(s',a')}_{\text{target}} - \underbrace{Q(s,a)}_{\text{predicción actual}}]$$

- $Q(s,a)$: Valor actual en la tabla
- $r + \gamma \max_{a'} Q(s',a')$: Valor objetivo (Bellman)
- $\alpha$: Controla qué tan rápido actualizamos

**2. DQN Loss**:
$$L(\theta) = (\underbrace{Q(s,a;\theta)}_{\text{red predice}} - \underbrace{r + \gamma \max_{a'} Q(s',a';\theta)}_{\text{target de Bellman}})^2$$

- Minimizamos la diferencia al cuadrado (MSE)
- La red aprende a predecir el target de Bellman

**3. Actualización de Pesos**:
$$\theta \leftarrow \theta - \alpha \nabla_\theta L(\theta)$$

- Movemos los pesos en dirección opuesta al gradiente
- PyTorch calcula $\nabla_\theta L(\theta)$ automáticamente con `.backward()`

---


<a name='1'></a>
## 1 - Preparación

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gymnasium as gym
from simple_dqn_tests import *

print(f"PyTorch version: {torch.__version__}")
print("✅ Todo listo!")

<a name='2'></a>
## 2 - Red Q Simple

Vamos a crear una red neuronal que tome un **estado** y devuelva **Q-values para cada acción**.

### Arquitectura
```
Estado (4 valores) → Capa Oculta (16 neuronas) → Q-values (2 acciones)
```

**Ejemplo**:
- Input: `[posición, velocidad, ángulo, vel_angular]`
- Output: `[Q(s, izquierda), Q(s, derecha)]`

<a name='ex-1'></a>
### Ejercicio 1 - crear_red_q

Crea una red Q simple con una capa oculta.

In [ ]:
# GRADED FUNCTION: crear_red_q

class RedQ(nn.Module):
    def __init__(self, estado_dim, accion_dim, oculta_dim=16):
        """
        Red Q simple.
        
        Argumentos:
        estado_dim -- dimensión del estado (ej: 4 para CartPole)
        accion_dim -- número de acciones (ej: 2 para CartPole)
        oculta_dim -- neuronas en capa oculta
        """
        super(RedQ, self).__init__()
        
        # (approx. 2 líneas)
        # 1. self.fc1 = nn.Linear(estado_dim, oculta_dim)
        # 2. self.fc2 = nn.Linear(oculta_dim, accion_dim)
        
        ### START CODE HERE ###
        
        
        ### END CODE HERE ###
    
    def forward(self, estado):
        """
        Forward pass: estado → Q-values
        """
        # (approx. 3 líneas)
        # 1. x = self.fc1(estado)
        # 2. x = torch.relu(x)
        # 3. q_values = self.fc2(x)
        
        ### START CODE HERE ###
        
        
        
        ### END CODE HERE ###
        
        return q_values

def crear_red_q(estado_dim, accion_dim):
    """Crea una instancia de RedQ."""
    ### START CODE HERE ###
    
    ### END CODE HERE ###
    return red

In [ ]:
# Prueba tu implementación
red = crear_red_q(estado_dim=4, accion_dim=2)
print(f"Red creada:")
print(red)

# Ejecuta el test
crear_red_q_test(crear_red_q)

<a name='3'></a>
## 3 - Predicción de Q-values

Ahora vamos a usar la red para **predecir Q-values** dado un estado.

<a name='ex-2'></a>
### Ejercicio 2 - predecir_q_values

Pasa un estado por la red y obtén Q-values.

In [ ]:
# GRADED FUNCTION: predecir_q_values

def predecir_q_values(red, estado):
    """
    Predice Q-values para un estado.
    
    Argumentos:
    red -- RedQ
    estado -- numpy array con el estado
    
    Retorna:
    q_values -- tensor con Q-values para cada acción
    """
    # (approx. 3 líneas)
    # 1. Convertir estado a tensor: torch.FloatTensor(estado)
    # 2. Pasar por la red: red(estado_tensor)
    # 3. Retornar q_values
    
    ### START CODE HERE ###
    
    
    
    ### END CODE HERE ###
    
    return q_values

In [ ]:
# Prueba tu implementación
red = crear_red_q(4, 2)
estado_test = np.array([0.1, 0.2, 0.3, 0.4])
q_vals = predecir_q_values(red, estado_test)

print(f"Estado: {estado_test}")
print(f"Q-values predichos: {q_vals}")
print(f"Mejor acción: {q_vals.argmax().item()}")

# Ejecuta el test
predecir_q_values_test(predecir_q_values, crear_red_q)

<a name='4'></a>
## 4 - Calcular Loss

Para entrenar la red, necesitamos calcular qué tan mal están las predicciones.

**Fórmula DQN**:
$$\text{Loss} = (Q(s,a) - \text{target})^2$$

Donde:
- $Q(s,a)$ = predicción de la red
- $\text{target} = r + \gamma \max_{a'} Q(s', a')$ = valor objetivo

<a name='ex-3'></a>
### Ejercicio 3 - calcular_dqn_loss

Calcula el loss entre predicción y target.

In [ ]:
# GRADED FUNCTION: calcular_dqn_loss

def calcular_dqn_loss(q_predicho, q_target):
    """
    Calcula MSE loss entre Q predicho y Q target.
    
    Argumentos:
    q_predicho -- Q-value predicho por la red
    q_target -- Q-value objetivo (calculado con Bellman)
    
    Retorna:
    loss -- MSE loss
    """
    # (approx. 2 líneas)
    # 1. criterion = nn.MSELoss()
    # 2. loss = criterion(q_predicho, q_target)
    
    ### START CODE HERE ###
    
    
    ### END CODE HERE ###
    
    return loss

In [ ]:
# Prueba tu implementación
q_pred = torch.tensor([1.0, 2.0])
q_targ = torch.tensor([1.5, 2.5])
loss = calcular_dqn_loss(q_pred, q_targ)

print(f"Q predicho: {q_pred}")
print(f"Q target: {q_targ}")
print(f"Loss: {loss.item():.4f}")

# Ejecuta el test
calcular_dqn_loss_test(calcular_dqn_loss)

<a name='5'></a>
## 5 - Entrenar un Paso

Un paso de entrenamiento DQN:
1. Predecir Q-values
2. Calcular target con ecuación de Bellman
3. Calcular loss
4. Backpropagation
5. Actualizar pesos

<a name='ex-4'></a>
### Ejercicio 4 - entrenar_paso_dqn

Implementa un paso de entrenamiento.

In [ ]:
# GRADED FUNCTION: entrenar_paso_dqn

def entrenar_paso_dqn(red, optimizer, estado, accion, recompensa, siguiente_estado, done, gamma=0.99):
    """
    Un paso de entrenamiento DQN.
    
    Argumentos:
    red -- RedQ
    optimizer -- optimizer de PyTorch
    estado -- estado actual
    accion -- acción tomada
    recompensa -- recompensa recibida
    siguiente_estado -- siguiente estado
    done -- si el episodio terminó
    gamma -- factor de descuento
    
    Retorna:
    loss -- loss del paso
    """
    # (approx. 8 líneas)
    # 1. optimizer.zero_grad()
    # 2. q_values = predecir_q_values(red, estado)
    # 3. q_actual = q_values[accion]
    # 4. Si done: q_target = recompensa
    #    Sino: q_target = recompensa + gamma * predecir_q_values(red, siguiente_estado).max()
    # 5. loss = calcular_dqn_loss(q_actual, q_target.detach())
    # 6. loss.backward()
    # 7. optimizer.step()
    
    ### START CODE HERE ###
    
    
    
    
    
    
    
    ### END CODE HERE ###
    
    return loss

In [ ]:
# Prueba tu implementación
red = crear_red_q(4, 2)
optimizer = optim.Adam(red.parameters(), lr=0.01)

estado = np.array([0.1, 0.2, 0.3, 0.4])
siguiente = np.array([0.2, 0.3, 0.4, 0.5])

print("Entrenando por 5 pasos...")
for i in range(5):
    loss = entrenar_paso_dqn(red, optimizer, estado, accion=0, recompensa=1.0, 
                             siguiente_estado=siguiente, done=False)
    if i % 2 == 0:
        print(f"Paso {i}: Loss = {loss.item():.4f}")

print("\n✅ La loss debería disminuir!")

# Ejecuta el test
entrenar_paso_dqn_test(entrenar_paso_dqn, crear_red_q)

<a name='6'></a>
## 6 - Loop de Entrenamiento

¡Ahora juntamos todo para entrenar un agente DQN en CartPole!

<a name='ex-5'></a>
### Ejercicio 5 - entrenar_dqn_simple

Implementa el loop de entrenamiento completo.

In [ ]:
# GRADED FUNCTION: entrenar_dqn_simple

def entrenar_dqn_simple(env, n_episodios=200, gamma=0.99, epsilon=0.1, lr=0.01):
    """
    Entrena un agente DQN simple.
    
    Argumentos:
    env -- ambiente de Gymnasium
    n_episodios -- número de episodios
    gamma -- factor de descuento
    epsilon -- probabilidad de exploración
    lr -- learning rate
    
    Retorna:
    red -- red entrenada
    recompensas -- lista de recompensas por episodio
    """
    # (approx. 12 líneas)
    # 1. Crear red y optimizer
    # 2. recompensas = []
    # 3. Para cada episodio:
    #    a. estado = env.reset()[0]
    #    b. recompensa_total = 0
    #    c. done = False
    #    d. Mientras not done:
    #       - Elegir acción (epsilon-greedy)
    #       - siguiente_estado, recompensa, terminated, truncated, _ = env.step(accion)
    #       - done = terminated or truncated
    #       - entrenar_paso_dqn(...)
    #       - recompensa_total += recompensa
    #       - estado = siguiente_estado
    #    e. recompensas.append(recompensa_total)
    
    ### START CODE HERE ###
    
    
    
    
    
    
    
    
    
    
    
    ### END CODE HERE ###
    
    return red, recompensas

In [ ]:
# Prueba tu implementación
print("Entrenando DQN simple en CartPole...")
print("Esto puede tomar 1-2 minutos...\n")

env = gym.make('CartPole-v1')
red_entrenada, recompensas = entrenar_dqn_simple(env, n_episodios=200)

print(f"\nPrimeros 10 episodios: {np.mean(recompensas[:10]):.1f} promedio")
print(f"Últimos 10 episodios: {np.mean(recompensas[-10:]):.1f} promedio")

env.close()

# Ejecuta el test
entrenar_dqn_simple_test(entrenar_dqn_simple)

## Visualizar Resultados

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(recompensas, alpha=0.6, label='Recompensa por episodio')
plt.plot(np.convolve(recompensas, np.ones(10)/10, mode='valid'), 
         linewidth=2, label='Promedio móvil (10)')
plt.xlabel('Episodio')
plt.ylabel('Recompensa Total')
plt.title('Entrenamiento DQN Simple en CartPole')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n🎯 Mejora: {recompensas[0]:.0f} → {np.mean(recompensas[-10:]):.0f}")

<a name='7'></a>
## 🎉 ¡Felicitaciones!

¡Has creado tu primer agente de Deep RL!

### Lo que lograste:

✅ Crear una red Q que aproxima Q-values

✅ Entrenar la red usando la ecuación de Bellman

✅ Resolver CartPole con DQN simple

✅ Entender cómo funciona DQN básico

### 🎯 Próximos Pasos

Ahora estás listo para DQN avanzado:
- `03_deep_rl/dqn/DQN_Exercise.ipynb` - DQN con replay buffer y target network
- `03_deep_rl/dqn/Double_DQN_Exercise.ipynb` - Mejora sobre DQN

---

### 💡 Conceptos Clave

**DQN Simple vs DQN Completo**:

| Aspecto | DQN Simple (este notebook) | DQN Completo |
|---------|---------------------------|-------------|
| Red neuronal | ✅ | ✅ |
| Replay buffer | ❌ | ✅ |
| Target network | ❌ | ✅ |
| Complejidad | Baja | Media |
| Rendimiento | Bueno | Excelente |

**¡Sigue aprendiendo! 🚀**